In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import re
import matplotlib.pyplot as plt
import numpy as np


/Users/jesucastin/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
root_dir="../../../data/dihedrals/raw_data"

In [3]:
# backbone PRE

# PRE
#psi
psi_pre_rot_s1_df=pd.read_csv('%s/preLC3B_OPLS/SIM_1/psi/preLC3B_OPLS_psi_rotamer.csv'%root_dir,
                         index_col=0)
psi_pre_rot_s2_df=pd.read_csv('%s/preLC3B_OPLS/SIM_2/psi/preLC3B_OPLS_psi_rotamer.csv'%root_dir,
                         index_col=0)
psi_pre_rot_s3_df=pd.read_csv('%s/preLC3B_OPLS/SIM_3/psi/preLC3B_OPLS_psi_rotamer.csv'%root_dir,
                         index_col=0)


psi_pre_rot_all=pd.concat([psi_pre_rot_s1_df, psi_pre_rot_s2_df,psi_pre_rot_s3_df],
                        ignore_index=True)


#phi
phi_pre_rot_s1_df=pd.read_csv('%s/preLC3B_OPLS/SIM_1/phi/preLC3B_OPLS_phi_rotamer.csv'%root_dir,
                         index_col=0)
phi_pre_rot_s2_df=pd.read_csv('%s/preLC3B_OPLS/SIM_2/phi/preLC3B_OPLS_phi_rotamer.csv'%root_dir,
                         index_col=0)
phi_pre_rot_s3_df=pd.read_csv('%s/preLC3B_OPLS/SIM_3/phi/preLC3B_OPLS_phi_rotamer.csv'%root_dir,
                         index_col=0)


phi_pre_rot_all=pd.concat([phi_pre_rot_s1_df, phi_pre_rot_s2_df,phi_pre_rot_s3_df],
                        ignore_index=True)

In [4]:
# bnd
#psi
psi_bnd_rot_s1_df=pd.read_csv('%s/LC3B_P62_INCLUDED/SIM_1/psi/LC3B_P62_INCLUDED_psi_rotamer.csv'%root_dir,
                         index_col=0)
psi_bnd_rot_s2_df=pd.read_csv('%s/LC3B_P62_INCLUDED/SIM_2/psi/LC3B_P62_INCLUDED_psi_rotamer.csv'%root_dir,
                         index_col=0)
psi_bnd_rot_s3_df=pd.read_csv('%s/LC3B_P62_INCLUDED/SIM_3/psi/LC3B_P62_INCLUDED_psi_rotamer.csv'%root_dir,
                         index_col=0)


psi_bnd_rot_all=pd.concat([psi_bnd_rot_s1_df, psi_bnd_rot_s2_df,psi_bnd_rot_s3_df],
                        ignore_index=True)


#phi
phi_bnd_rot_s1_df=pd.read_csv('%s/LC3B_P62_INCLUDED/SIM_1/phi/LC3B_P62_INCLUDED_phi_rotamer.csv'%root_dir,
                         index_col=0)
phi_bnd_rot_s2_df=pd.read_csv('%s/LC3B_P62_INCLUDED/SIM_2/phi/LC3B_P62_INCLUDED_phi_rotamer.csv'%root_dir,
                         index_col=0)
phi_bnd_rot_s3_df=pd.read_csv('%s/LC3B_P62_INCLUDED/SIM_3/phi/LC3B_P62_INCLUDED_phi_rotamer.csv'%root_dir,
                         index_col=0)


phi_bnd_rot_all=pd.concat([phi_bnd_rot_s1_df, phi_bnd_rot_s2_df,phi_bnd_rot_s3_df],
                        ignore_index=True)

In [5]:
def num_ext_BB(DF,angle_type):
    FORM_dih_sel_col=[]
    for i in DF['RESIDUES']:
        j=i.split('.')[0].replace(angle_type,'')
        t_ls=re.findall(r'\d+', j)
        t=int(''.join(t_ls))
        FORM_dih_sel_col.append(t)
    return(FORM_dih_sel_col)


def rotamer_shift_BB(f1_rotamers, f2_rotamers):
    df1=abs(f1_rotamers[0]-f2_rotamers[0])
    df2=abs(f1_rotamers[1]-f2_rotamers[1])
#     df3=abs(f1_rotamers[2]-f2_rotamers[2])
    
    change=(df1+df2)/2
    
    return(change)




def bb_rot_frac(system_rot_all):
    
    system_all_sims_rot_states=[]
    res_ls_system=[]

    for resid_ind in range(system_rot_all.shape[1]):
        res_ls_system.append(system_rot_all.columns[resid_ind])

        #BB HAS CIS AND TRAS rotamer states

        rotameric_states_ls=system_rot_all[system_rot_all.columns[resid_ind]].to_list()
        f_cis=rotameric_states_ls.count(0)/len(rotameric_states_ls)
        f_trans=rotameric_states_ls.count(1)/len(rotameric_states_ls)

        system_all_sims_rotameric_states_l=[f_cis,f_trans]
        system_all_sims_rot_states.append(system_all_sims_rotameric_states_l)

    system_all_sims_SC_rotameric_states=pd.DataFrame()
    system_all_sims_SC_rotameric_states['RESIDUES']=res_ls_system
    system_all_sims_SC_rotameric_states['f(Cis)']=[i[0] for i in system_all_sims_rot_states]
    system_all_sims_SC_rotameric_states['f(Trans)']=[i[1] for i in system_all_sims_rot_states]
    
    return system_all_sims_SC_rotameric_states
    
    
def bb_rot_shift_compute(pre_rot_state_frac, unb_rot_state_frac, angle_type_): # pre, unb
    F_shift=[]
    for i in range(1,120):
        if i in num_ext_BB(unb_rot_state_frac, angle_type_):
            PRE=pre_rot_state_frac[pre_rot_state_frac['Rn']==i]
            PRE_ROT=[PRE['f(Cis)'].iloc[0],PRE['f(Trans)'].iloc[0]]

            UNB=unb_rot_state_frac[unb_rot_state_frac['Rn']==i]
            UNB_ROT=[UNB['f(Cis)'].iloc[0],UNB['f(Trans)'].iloc[0]]

            F_shift.append(rotamer_shift_BB(PRE_ROT,UNB_ROT))

        else:
            F_shift.append(0)
    
    F_shift_df=pd.DataFrame()
    F_shift_df['Residues']=[i for i in range(1,120)]
    F_shift_df['Shift']=F_shift
    
    return F_shift_df

In [7]:
psi_pre_rot_fracr=bb_rot_frac(psi_pre_rot_all)
phi_pre_rot_fracr=bb_rot_frac(phi_pre_rot_all)

psi_bnd_rot_fracr=bb_rot_frac(psi_bnd_rot_all)
phi_bnd_rot_fracr=bb_rot_frac(phi_bnd_rot_all)


psi_pre_rot_fracr['Rn']=num_ext_BB(psi_pre_rot_fracr, "psi")
phi_pre_rot_fracr['Rn']=num_ext_BB(phi_pre_rot_fracr, "phi")

psi_bnd_rot_fracr['Rn']=num_ext_BB(psi_bnd_rot_fracr, "psi")
phi_bnd_rot_fracr['Rn']=num_ext_BB(phi_bnd_rot_fracr, "phi")



psi_shift_pre_vs_bnd=bb_rot_shift_compute(psi_pre_rot_fracr, psi_bnd_rot_fracr, "psi")
phi_shift_pre_vs_bnd=bb_rot_shift_compute(phi_pre_rot_fracr, phi_bnd_rot_fracr, "phi")

In [8]:
def num_ext_BB(DF,angle_type):
    FORM_dih_sel_col=[]
    for i in DF['RESIDUES']:
        j=i.split('.')[0].replace(angle_type,'')
        t_ls=re.findall(r'\d+', j)
        t=int(''.join(t_ls))
        FORM_dih_sel_col.append(t)
    return(FORM_dih_sel_col)


def rotamer_shift_BB(f1_rotamers, f2_rotamers):
    df1=abs(f1_rotamers[0]-f2_rotamers[0])
    df2=abs(f1_rotamers[1]-f2_rotamers[1])
#     df3=abs(f1_rotamers[2]-f2_rotamers[2])
    
    change=(df1+df2)/2
    
    return(change)




def bb_rot_frac(system_rot_all):
    
    system_all_sims_rot_states=[]
    res_ls_system=[]

    for resid_ind in range(system_rot_all.shape[1]):
        res_ls_system.append(system_rot_all.columns[resid_ind])

        #BB HAS CIS AND TRAS rotamer states

        rotameric_states_ls=system_rot_all[system_rot_all.columns[resid_ind]].to_list()
        f_cis=rotameric_states_ls.count(0)/len(rotameric_states_ls)
        f_trans=rotameric_states_ls.count(1)/len(rotameric_states_ls)

        system_all_sims_rotameric_states_l=[f_cis,f_trans]
        system_all_sims_rot_states.append(system_all_sims_rotameric_states_l)

    system_all_sims_SC_rotameric_states=pd.DataFrame()
    system_all_sims_SC_rotameric_states['RESIDUES']=res_ls_system
    system_all_sims_SC_rotameric_states['f(Cis)']=[i[0] for i in system_all_sims_rot_states]
    system_all_sims_SC_rotameric_states['f(Trans)']=[i[1] for i in system_all_sims_rot_states]
    
    return system_all_sims_SC_rotameric_states
    
    
def bb_rot_shift_compute(pre_rot_state_frac, unb_rot_state_frac, angle_type_): # pre, unb
    F_shift=[]
    for i in range(1,120):
        if i in num_ext_BB(unb_rot_state_frac, angle_type_):
            PRE=pre_rot_state_frac[pre_rot_state_frac['Rn']==i]
            PRE_ROT=[PRE['f(Cis)'].iloc[0],PRE['f(Trans)'].iloc[0]]

            UNB=unb_rot_state_frac[unb_rot_state_frac['Rn']==i]
            UNB_ROT=[UNB['f(Cis)'].iloc[0],UNB['f(Trans)'].iloc[0]]

            F_shift.append(rotamer_shift_BB(PRE_ROT,UNB_ROT))

        else:
            F_shift.append(0)
    
    F_shift_df=pd.DataFrame()
    F_shift_df['Residues']=[i for i in range(1,120)]
    F_shift_df['Shift']=F_shift
    
    return F_shift_df

In [11]:
psi_pre_rot_fracr=bb_rot_frac(psi_pre_rot_all)
phi_pre_rot_fracr=bb_rot_frac(phi_pre_rot_all)

psi_bnd_rot_fracr=bb_rot_frac(psi_bnd_rot_all)
phi_bnd_rot_fracr=bb_rot_frac(phi_bnd_rot_all)


psi_pre_rot_fracr['Rn']=num_ext_BB(psi_pre_rot_fracr, "psi")
phi_pre_rot_fracr['Rn']=num_ext_BB(phi_pre_rot_fracr, "phi")

psi_bnd_rot_fracr['Rn']=num_ext_BB(psi_bnd_rot_fracr, "psi")
phi_bnd_rot_fracr['Rn']=num_ext_BB(phi_bnd_rot_fracr, "phi")



psi_shift_pre_vs_bnd=bb_rot_shift_compute(psi_pre_rot_fracr, psi_bnd_rot_fracr, "psi")
phi_shift_pre_vs_bnd=bb_rot_shift_compute(phi_pre_rot_fracr, phi_bnd_rot_fracr, "phi")

In [12]:
shift_type=[]
max_shift_ls=[]
for i,j,k in zip(psi_shift_pre_vs_bnd['Shift'],
              phi_shift_pre_vs_bnd['Shift'],
                phi_shift_pre_vs_bnd['Residues']):
    
    max_shift=max(i,j)
    max_shift_ls.append(max_shift)
    if max_shift ==i:
        shift_type.append('psi')
    elif max_shift==j:
        shift_type.append('phi')

In [13]:
bb_pre_vs_bnd_shift_data=pd.DataFrame()
bb_pre_vs_bnd_shift_data['Residues']=psi_shift_pre_vs_bnd['Residues']
bb_pre_vs_bnd_shift_data['psi shift']=psi_shift_pre_vs_bnd['Shift']
bb_pre_vs_bnd_shift_data['phi shift']=phi_shift_pre_vs_bnd['Shift']
bb_pre_vs_bnd_shift_data['Max']=max_shift_ls
bb_pre_vs_bnd_shift_data['Max BB']=shift_type
bb_pre_vs_bnd_shift_data

,Residues,psi shift,phi shift,Max,Max BB
0,1,0.344322,0.022977,0.344322,psi
1,2,0.318348,0.000000,0.318348,psi
2,3,0.200799,0.150516,0.200799,psi
3,4,0.161838,0.019314,0.161838,psi
4,5,0.042624,0.011655,0.042624,psi
...,...,...,...,...,...
114,115,0.073260,0.000000,0.073260,psi
115,116,0.110223,0.000000,0.110223,psi
116,117,0.386613,0.000000,0.386613,psi
117,118,0.182817,0.026640,0.182817,psi


In [15]:
bb_pre_vs_bnd_shift_data.to_csv("../../../data/dihedrals/rotamer_shift/BACKBONE_ROTAMER_SHIFT_PRE_VS_BND.csv")